# RN-dS 静态 junction 图像生成流程讲解

这个 notebook 讲的是本项目中 **Reissner-Nordstrom-de Sitter, RN-dS** 静态薄壳 junction 图像从参数到 PNG 的完整运行逻辑。

读法建议：先顺着 Markdown 看一遍整体调用链，再按需运行代码单元。代码单元尽量设计成轻量检查，不会默认重新生成整个 atlas。

## 一句话总览

RN-dS 图像不是一个单独 renderer，而是 atlas 生成流程中的一个 `family="rnds"` 分支：

1. `scripts/generate_junction_atlas.py` 扫描 RN/RN-dS 参数网格。
2. `junction_analysis.py` 把参数变成内外两个 metric，并做物理筛选。
3. `junction_tracing.py` 对每个屏幕半径追一条 null ray，记录壳穿越和 ray segment。
4. `transfer.py` 找 ray 与薄盘的交点。
5. `junctions.py` 给交点加 junction 红移因子。
6. `sources.py` 把发射率和红移合成观测强度。
7. `imaging.py` 把一维径向强度剖面旋转成二维轴对称图像。

主入口是 `scripts/generate_junction_atlas.py`；较小的通用演示入口是 `scripts/generate_static_junction_profiles.py`。

In [ ]:
from pathlib import Path
import csv
import json
import math
import sys

# 让 notebook 无论从仓库根目录还是 notebooks/ 目录启动，都能找到项目根目录。
ROOT = Path.cwd()
if not (ROOT / "src" / "spherical_raytracing").exists():
    # 如果当前目录是 notebooks/，项目根目录就在上一级。
    ROOT = ROOT.parent

SRC = ROOT / "src"
SCRIPTS = ROOT / "scripts"
# 把 src/ 和 scripts/ 加入 import 路径，后面可以直接 import 项目代码和脚本辅助函数。
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("repo root:", ROOT)
print("src exists:", SRC.exists())
print("outputs/junction_atlas exists:", (ROOT / "outputs" / "junction_atlas").exists())

## 相关文件地图

下面这些文件构成 RN-dS 图像路径。可以把它分成三层：

- **脚本层**：决定扫描哪些参数、写哪些 PNG/CSV/JSON。
- **物理/追光层**：定义 RN-dS metric、junction、壳匹配、ray tracing。
- **成像层**：薄盘交点、红移权重、径向 profile、二维图像。

In [ ]:
# 这些文件就是 RN-dS 图像链路的主要节点：脚本入口、物理模型、追光、成像。
key_files = [
    "scripts/generate_junction_atlas.py",
    "scripts/generate_static_junction_profiles.py",
    "scripts/write_junction_atlas_report.py",
    "src/spherical_raytracing/metrics.py",
    "src/spherical_raytracing/junction_analysis.py",
    "src/spherical_raytracing/junctions.py",
    "src/spherical_raytracing/junction_tracing.py",
    "src/spherical_raytracing/transfer.py",
    "src/spherical_raytracing/sources.py",
    "src/spherical_raytracing/imaging.py",
    "src/spherical_raytracing/observers.py",
]

for rel in key_files:
    path = ROOT / rel
    print(f"{'OK' if path.exists() else 'missing'}  {rel}")

## 1. RN-dS metric 本身

RN-dS metric 在 `metrics.py` 里由 `ReissnerNordstromDeSitterMetric` 定义。项目使用静态球对称形式：

`ds^2 = -A(r) dt^2 + B(r) dr^2 + r^2 dOmega^2`

RN-dS 的核心函数是：

`A(r) = 1 - 2M/r + Q^2/r^2 - Lambda r^2/3`

代码还提供：

- `horizons()`：求正实视界根。
- `static_domains()`：找 `A(r)>0` 的静态区间。
- `valid_radial_domain()`：追光时使用的有效径向边界。
- `photon_spheres()` / `critical_curves()`：找 photon sphere 和临界屏幕半径。

### 代码复核结论

我对 `metrics.py` 里的 RN-dS 公式逐项核对了一遍：当前实现采用的是标准静态坐标规范 `B(r)=1/A(r)`，所以 `1/B(r)` 理论上应该和 `A(r)` 完全相同。代码中的几个关键式子是自洽的：

- `A(r) = 1 - 2M/r + Q^2/r^2 - Lambda r^2/3`：RN-dS 标准形式。
- `B(r) = 1/A(r)`：与 Schwarzschild、RN、de Sitter 的常用静态坐标一致。
- `dA_dr = 2M/r^2 - 2Q^2/r^3 - 2 Lambda r/3`：导数正确。
- `G(u,b) = 1/b^2 - u^2 + 2Mu^3 - Q^2u^4 + Lambda/3`：由 `G=1/b^2-u^2 A(1/u)` 得到，符号正确。
- photon sphere 方程中 `Lambda` 项会抵消，所以 RN-dS 的 photon sphere 半径与 RN 同式：`r=(3M ± sqrt(9M^2-8Q^2))/2`。

因此从公式层面看，当前 RN-dS 度规实现没有明显问题。下面用图和数值检查再验证一遍。

In [ ]:
from spherical_raytracing.metrics import ReissnerNordstromDeSitterMetric
from spherical_raytracing.junctions import clean_rnds_static_patch

# 先单独看一个外侧 RN-dS metric，理解它自己的视界、静态区和 photon sphere。
metric = ReissnerNordstromDeSitterMetric(
    mass=1.0,
    charge=0.2,
    cosmological_constant=0.01,
    region="outer",
)

# A(r)>0 的地方才是静态区域；RN-dS 通常有黑洞视界和宇宙学视界。
print("A(5) =", metric.A(5.0))
print("horizons =", metric.horizons())
print("static_domains =", metric.static_domains())
print("valid_radial_domain =", metric.valid_radial_domain())
print("clean_rnds_static_patch =", clean_rnds_static_patch(metric))
print("photon_spheres =", metric.photon_spheres())
print("critical_curves =", metric.critical_curves())

### 画出四种度规的 `A(r)` 和 `1/B(r)`

下面两张图使用同一组参数尺度：`M=1`、`Q=0.5`、`Lambda=0.01`。项目里没有单独的 pure de Sitter metric 类，所以 de Sitter 曲线在 notebook 中用解析式临时定义：

`A_dS(r)=1-Lambda r^2/3`, `B_dS(r)=1/A_dS(r)`。

因为这四个静态坐标度规都满足 `B=1/A`，所以第二张 `1/B(r)` 图应该与第一张 `A(r)` 图重合；这正是一个很直接的 sanity check。

In [ ]:
import math
import numpy as np

from spherical_raytracing.metrics import ReissnerNordstromMetric, SchwarzschildMetric

M = 1.0
Q = 0.5
Lambda = 0.01

schwarzschild = SchwarzschildMetric(mass=M)
rn = ReissnerNordstromMetric(mass=M, charge=Q)
rnds = ReissnerNordstromDeSitterMetric(mass=M, charge=Q, cosmological_constant=Lambda)

# pure de Sitter 没有项目类，这里只为对照图临时写解析函数。
def ds_A(r: float) -> float:
    return 1.0 - Lambda * r**2 / 3.0

def ds_B(r: float) -> float:
    a = ds_A(r)
    return math.inf if a == 0.0 else 1.0 / a

def evaluate(fn, radii):
    # 项目里的 A/B 方法是标量函数，所以用列表推导逐点求值。
    return np.array([fn(float(r)) for r in radii], dtype=float)

def one_over_B(metric_or_B, radii):
    values = evaluate(metric_or_B, radii)
    return np.where(np.isfinite(values) & (values != 0.0), 1.0 / values, 0.0)

# 避开 r=0 奇点；上界越过 dS/RN-dS 的宇宙学视界，方便看 A 过零。
radii = np.linspace(0.35, 22.0, 1600)

curves = {
    "Schwarzschild": (lambda r: schwarzschild.A(r), lambda r: schwarzschild.B(r)),
    "RN": (lambda r: rn.A(r), lambda r: rn.B(r)),
    "de Sitter": (ds_A, ds_B),
    "RN-dS": (lambda r: rnds.A(r), lambda r: rnds.B(r)),
}

print("Schwarzschild horizons:", schwarzschild.horizons())
print("RN horizons:", rn.horizons())
print("de Sitter cosmological horizon:", math.sqrt(3.0 / Lambda))
print("RN-dS horizons:", rnds.horizons())

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(7.2, 4.4))
for label, (A_fn, _B_fn) in curves.items():
    axis.plot(radii, evaluate(A_fn, radii), label=label, linewidth=1.6)

axis.axhline(0.0, color="black", linewidth=0.8, alpha=0.55)
axis.set_xlabel("r")
axis.set_ylabel("A(r)")
axis.set_title("Metric function A(r)")
axis.set_ylim(-1.2, 2.0)
axis.grid(True, alpha=0.25, linewidth=0.6)
axis.legend()
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(7.2, 4.4))
for label, (_A_fn, B_fn) in curves.items():
    axis.plot(radii, one_over_B(B_fn, radii), label=label, linewidth=1.6)

axis.axhline(0.0, color="black", linewidth=0.8, alpha=0.55)
axis.set_xlabel("r")
axis.set_ylabel("1 / B(r)")
axis.set_title("Inverse radial metric function 1/B(r)")
axis.set_ylim(-1.2, 2.0)
axis.grid(True, alpha=0.25, linewidth=0.6)
axis.legend()
plt.show()

### 画出光子球诊断函数 `F(r)` 和 `b^2(r)`

静态球对称度规的圆形 null orbit 条件可以写成：

`F(r) = r A'(r) - 2 A(r) = 0`

等价地，也可以看：

`b^2(r) = r^2 / A(r)`

`b^2(r)` 的极值点就是 photon sphere，对应临界冲量参数 `b_crit = sqrt(b^2(r_ph))`。注意只有 `A(r)>0` 的静态区域里，`b^2` 才适合作为这个意义下的实数屏幕冲量参数。

对 pure de Sitter，`F(r)=-2`，所以不会有有限半径 photon sphere；对 RN-dS，`Lambda` 项在 `F(r)` 中抵消，所以 photon sphere 半径和同参数 RN 一样，但 `b_crit` 会因为 `A(r_ph)` 含 `Lambda` 而改变。

In [ ]:
def ds_dA_dr(r: float) -> float:
    return -2.0 * Lambda * r / 3.0

def photon_F(A_fn, dA_fn, radii):
    # F(r)=0 是圆形光子轨道条件；这里仍用逐点求值保持和项目标量 API 一致。
    return np.array([float(r) * dA_fn(float(r)) - 2.0 * A_fn(float(r)) for r in radii], dtype=float)

def b_squared(A_fn, radii):
    A_values = evaluate(A_fn, radii)
    # 只在静态区 A>0 画 b^2；非静态区或视界附近用 NaN 断开曲线。
    return np.where(A_values > 1e-8, radii**2 / A_values, np.nan)

photon_curves = {
    "Schwarzschild": (lambda r: schwarzschild.A(r), lambda r: schwarzschild.dA_dr(r), schwarzschild.photon_spheres()),
    "RN": (lambda r: rn.A(r), lambda r: rn.dA_dr(r), rn.photon_spheres()),
    "de Sitter": (ds_A, ds_dA_dr, []),
    "RN-dS": (lambda r: rnds.A(r), lambda r: rnds.dA_dr(r), rnds.photon_spheres()),
}

for label, (A_fn, _dA_fn, photon_spheres) in photon_curves.items():
    print(label, "photon_spheres =", photon_spheres)
    for r_ph in photon_spheres:
        print("  r_ph =", r_ph, "b_crit =", math.sqrt(r_ph**2 / A_fn(r_ph)))

In [ ]:
figure, axis = plt.subplots(figsize=(7.2, 4.4))
for label, (A_fn, dA_fn, photon_spheres) in photon_curves.items():
    axis.plot(radii, photon_F(A_fn, dA_fn, radii), label=label, linewidth=1.6)
    # 用竖线标出项目代码返回的 photon sphere 位置，方便和 F=0 交点对照。
    for r_ph in photon_spheres:
        axis.axvline(r_ph, color="0.35", linestyle=":", linewidth=0.8, alpha=0.65)

axis.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
axis.set_xlabel("r")
axis.set_ylabel("F(r) = r A'(r) - 2 A(r)")
axis.set_title("Photon-sphere condition F(r)=0")
axis.set_ylim(-4.0, 6.0)
axis.grid(True, alpha=0.25, linewidth=0.6)
axis.legend()
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(7.2, 4.4))
for label, (A_fn, _dA_fn, photon_spheres) in photon_curves.items():
    axis.plot(radii, b_squared(A_fn, radii), label=label, linewidth=1.6)
    for r_ph in photon_spheres:
        b2_ph = r_ph**2 / A_fn(r_ph)
        axis.scatter([r_ph], [b2_ph], s=28, zorder=3)

axis.set_xlabel("r")
axis.set_ylabel("b^2(r) = r^2 / A(r)")
axis.set_title("Impact-parameter function b^2(r)")
# 视界附近 b^2 会发散；截断 y 轴让光子球附近的极值看得清楚。
axis.set_ylim(0.0, 80.0)
axis.grid(True, alpha=0.25, linewidth=0.6)
axis.legend()
plt.show()

### 除了画图，还可以怎么验证？

画图能看整体形状和视界过零，但更可靠的是做代数极限和数值残差检查：

1. **退化极限**：`Lambda=0` 时 RN-dS 必须退化成 RN；`Q=0` 时必须退化成 Schwarzschild-de Sitter；`M=Q=0` 会是 pure de Sitter，不过当前项目类要求 `mass>0`，所以 pure dS 需要单独类或局部解析函数。
2. **`B=1/A` 恒等式**：在非视界点检查 `A(r) - 1/B(r)` 是否为 0。
3. **视界残差**：`horizons()` 返回的每个根都应满足 `A(r_h)=0`。
4. **静态区符号**：`static_domains()` 每个区间中点应满足 `A(r)>0`，区间外相邻区域通常 `A(r)<0`。
5. **导数检查**：用有限差分检查 `dA_dr()`。
6. **photon sphere 条件**：每个 photon sphere 应满足 `r*A'(r)-2*A(r)=0`，并且 RN-dS 中这个条件不依赖 `Lambda`。
7. **有效势检查**：`G(u,b)` 应满足 `G = 1/b^2 - u^2*A(1/u)`。
8. **端到端追光检查**：低于临界冲量参数的 ray 应落入黑洞视界；高于临界值的 outbound ray 在 RN-dS 中应终止在宇宙学视界。项目里已经有 `tests/test_metrics.py` 和 `tests/test_solvers_bounded_domain.py` 覆盖这些点。

In [ ]:
# 一组轻量数值验证：这些检查比肉眼看图更适合防止符号和因子写错。
rn_limit = ReissnerNordstromDeSitterMetric(mass=M, charge=Q, cosmological_constant=0.0)
check_radii = np.array([0.8, 2.5, 6.0, 12.0])

# Lambda=0 极限：RN-dS -> RN。
lambda_zero_error = np.max(np.abs(evaluate(rn_limit.A, check_radii) - evaluate(rn.A, check_radii)))

# B=1/A：避开视界附近，检查 A 和 1/B 的最大偏差。
patch = clean_rnds_static_patch(rnds)
r_left, r_right = patch
static_radii = np.linspace(r_left + 0.2, r_right - 0.2, 20)
inverse_B_error = np.max(np.abs(evaluate(rnds.A, static_radii) - one_over_B(rnds.B, static_radii)))

# horizons() 根残差。
horizon_residual = max(abs(rnds.A(root)) for root in rnds.horizons())

# dA/dr 有限差分残差。
def finite_difference_dA(metric, r, h=1e-5):
    return (metric.A(r + h) - metric.A(r - h)) / (2.0 * h)

derivative_error = max(abs(rnds.dA_dr(float(r)) - finite_difference_dA(rnds, float(r))) for r in static_radii[::4])

# photon sphere 条件 r A'(r)-2A(r)=0。
photon_residual = max(abs(r * rnds.dA_dr(r) - 2.0 * rnds.A(r)) for r in rnds.photon_spheres())

# G(u,b) 是否等于 1/b^2 - u^2 A(1/u)。
u = 1.0 / 6.0
b = 5.0
G_expected = 1.0 / b**2 - u**2 * rnds.A(1.0 / u)
G_error = abs(rnds.G(u, b) - G_expected)

print("Lambda=0 RN limit max error:", lambda_zero_error)
print("A - 1/B max error in clean static patch:", inverse_B_error)
print("max |A(horizon)|:", horizon_residual)
print("dA/dr finite-difference max error:", derivative_error)
print("photon-sphere equation max residual:", photon_residual)
print("G potential identity error:", G_error)

## 2. Atlas 参数如何变成一个 RN-dS junction

`junction_analysis.py` 负责把抽象参数变成可追光的对象。关键数据结构是 `AtlasParameters`：

- `family="rnds"` 表示内外都是 RN-dS metric。
- `m_minus, q_minus, lambda_minus` 是壳内侧参数。
- `m_plus, q_plus, lambda_plus` 是壳外侧参数。
- `shell_radius` 是薄壳半径。
- `observer_radius` 是外侧有限半径静态观察者位置。

`build_junction(parameters)` 会创建：

- `StaticJunctionSpacetime(inner_metric, outer_metric, StaticShell(...))`
- `FiniteStaticObserver(r_obs=observer_radius, metric=outer_metric)`

RN-dS 的额外要求是：壳和观测者都必须落在干净的 black-hole-to-cosmological static patch 里，也就是外黑洞视界与宇宙学视界之间。

In [ ]:
from spherical_raytracing.junction_analysis import AtlasParameters, admit_parameters, build_junction, classify_photon_spheres

# 这组参数对应一个 RN-dS/RN-dS 静态薄壳：minus 是壳内侧，plus 是壳外侧。
params = AtlasParameters(
    family="rnds",
    m_minus=0.8,
    m_plus=1.0,
    q_minus=0.2,
    q_plus=0.2,
    lambda_minus=0.01,
    lambda_plus=0.01,
    shell_radius=5.0,
    observer_radius=10.0,
)

# admission 是 atlas 的低成本筛选结果；build_junction 才创建真正用于追光的对象。
admission = admit_parameters(params)
junction, observer = build_junction(params)

print("admitted:", admission.admitted)
print("rejection_reasons:", admission.rejection_reasons)
print("warnings:", admission.warnings)
print("surface_energy_density:", admission.surface_energy_density)
print("surface_pressure:", admission.surface_pressure)
print("lambda_jump:", admission.lambda_jump)
print("photon_sphere_classification:", classify_photon_spheres(params))
print("observer A(r_obs):", observer.metric.A(observer.r_obs))
print("junction warnings:", junction.physics_diagnostics(observer.r_obs).warnings)

### 画出 RN-dS junction 的内外曲线和连续拼接后的曲线

这里正是 thin-shell junction 最容易画错的地方。论文并不是把两边的 Schwarzschild/RN-dS 坐标时间 `t_-`、`t_+` 当成同一个全局时间来硬接；它先在壳面 `r=R(τ)` 上要求诱导度规连续 `[h_ab]=0`，并在壳附近引入 Gaussian normal coordinates。法向导数可以跳变，这个跳变对应壳的面能量和面压力。

对静态壳 `Rdot=0`，壳面 proper time 满足 `dτ^2 = A_-(R) dt_-^2 = A_+(R) dt_+^2`。所以如果选择外侧时间 `T=t_+` 作为图中的公共时间，内侧 lapse 必须重标定为

`A_-^eff(r) = A_-(r) A_+(R_shell) / A_-(R_shell)`，外侧保持 `A_+^eff(r)=A_+(r)`。

这样 `A_eff` 在壳面连续；原始的 `A_-(R_shell)` 和 `A_+(R_shell)` 可以不同，因为它们属于两套不同的静态坐标时间。径向项也类似，不能直接用同一个 areal-`r` 坐标下的 `B_-(R)`、`B_+(R)` 判断完整度规是否连续；论文是在 Gaussian normal coordinate 中保证完整 metric 连续。

论文后面推光线穿壳时也正是用了这个坐标基底变换。静态壳下，能量和冲量参数变换为 `E_to/E_from=sqrt(A_to(R_shell)/A_from(R_shell))`、`b_to=b_from sqrt(A_from(R_shell)/A_to(R_shell))`，角动量 `L` 保持不变。本项目 `junctions.py::match_static_shell()` 实现的就是这个关系。所以这里的问题不是 solver 没做 Israel matching，而是如果直接画 raw `A_-`/`A_+` 的分段图，会把两套时间坐标误读成同一套时间坐标。

下面两张图中：虚线/点划线是内外 RN-dS 参数在各自坐标中的原始延拓曲线，粗实线是采用外侧时间归一化后的 junction 曲线。`F(r)=rA'(r)-2A(r)` 在内侧也乘同一个常数，所以 photon sphere 的零点位置不变，但 `F` 本身允许在壳面跳变。

In [ ]:
inner_metric = junction.inner_metric
outer_metric = junction.outer_metric
R_shell = params.shell_radius

# 选一个覆盖壳内外、又不太贴近 r=0 奇点的半径窗口。
r_junction = np.linspace(0.35, min(params.observer_radius * 1.4, 16.0), 1400)
inner_mask = r_junction <= R_shell
outer_mask = r_junction >= R_shell

# 论文的静态壳匹配相当于在壳面把两侧 proper time 对齐。
# 这里采用外侧时间归一化，所以内侧 lapse A 需要乘这个常数。
A_inner_shell = inner_metric.A(R_shell)
A_outer_shell = outer_metric.A(R_shell)
inner_time_scale = A_outer_shell / A_inner_shell

def piecewise_values(inner_fn, outer_fn, radii):
    # 分段取值：壳内用 inner，壳外用 outer。inner_fn 可以是已经重标定后的函数。
    return np.where(
        radii <= R_shell,
        evaluate(inner_fn, radii),
        evaluate(outer_fn, radii),
    )

def mark_photon_spheres(axis, metric, label_prefix, color):
    for r_ph in metric.photon_spheres():
        if r_junction[0] <= r_ph <= r_junction[-1]:
            axis.axvline(r_ph, color=color, linestyle=":", linewidth=0.9, alpha=0.65)
            axis.text(r_ph, axis.get_ylim()[1] * 0.88, f"{label_prefix} ph", rotation=90, va="top", ha="right", fontsize=8, color=color)

print("shell radius:", R_shell)
print("inner metric parameters:", inner_metric)
print("outer metric parameters:", outer_metric)
print("inner horizons:", inner_metric.horizons())
print("outer horizons:", outer_metric.horizons())
print("inner photon spheres:", inner_metric.photon_spheres())
print("outer photon spheres:", outer_metric.photon_spheres())
print("raw A_inner(R_shell):", A_inner_shell)
print("raw A_outer(R_shell):", A_outer_shell)
print("inner time/lapse scale A_outer(R)/A_inner(R):", inner_time_scale)
print("scaled A_inner_eff(R_shell):", inner_time_scale * A_inner_shell)

In [ ]:
figure, axis = plt.subplots(figsize=(7.4, 4.5))

inner_A_full = evaluate(inner_metric.A, r_junction)
outer_A_full = evaluate(outer_metric.A, r_junction)
inner_A_eff = inner_time_scale * inner_A_full
junction_A_eff = piecewise_values(lambda r: inner_time_scale * inner_metric.A(r), outer_metric.A, r_junction)

axis.plot(r_junction, inner_A_full, linestyle="--", linewidth=1.0, label="raw inner A_-(t_-)", color="tab:blue", alpha=0.55)
axis.plot(r_junction, outer_A_full, linestyle="-.", linewidth=1.0, label="raw outer A_+(t_+)", color="tab:orange", alpha=0.75)
axis.plot(r_junction[inner_mask], junction_A_eff[inner_mask], linewidth=2.3, color="tab:blue", label="matched A_eff: inner")
axis.plot(r_junction[outer_mask], junction_A_eff[outer_mask], linewidth=2.3, color="tab:orange", label="matched A_eff: outer")
axis.scatter([R_shell, R_shell], [A_inner_shell, A_outer_shell], color=["tab:blue", "tab:orange"], s=28, zorder=5, label="raw shell values")
axis.scatter([R_shell], [inner_time_scale * A_inner_shell], color="black", marker="x", s=46, zorder=6, label="matched shell value")

axis.axhline(0.0, color="black", linewidth=0.8, alpha=0.55)
axis.axvline(R_shell, color="black", linestyle="--", linewidth=1.0, alpha=0.8, label="shell")
axis.set_xlabel("r")
axis.set_ylabel("A(r) or shell-normalized A_eff(r)")
axis.set_title("RN-dS junction lapse: raw coordinates vs matched shell time")
axis.set_ylim(-1.0, 1.6)
axis.grid(True, alpha=0.25, linewidth=0.6)
mark_photon_spheres(axis, inner_metric, "inner", "tab:blue")
mark_photon_spheres(axis, outer_metric, "outer", "tab:orange")
axis.legend(fontsize=8, loc="best")
plt.show()

In [ ]:
def metric_F(metric, radii):
    return photon_F(metric.A, metric.dA_dr, radii)

figure, axis = plt.subplots(figsize=(7.4, 4.5))

inner_F_full = metric_F(inner_metric, r_junction)
outer_F_full = metric_F(outer_metric, r_junction)
inner_F_eff = inner_time_scale * inner_F_full
junction_F_eff = piecewise_values(lambda r: inner_time_scale * (r * inner_metric.dA_dr(r) - 2.0 * inner_metric.A(r)), lambda r: r * outer_metric.dA_dr(r) - 2.0 * outer_metric.A(r), r_junction)

axis.plot(r_junction, inner_F_full, linestyle="--", linewidth=1.0, label="raw inner F_-(t_-)", color="tab:blue", alpha=0.55)
axis.plot(r_junction, outer_F_full, linestyle="-.", linewidth=1.0, label="raw outer F_+(t_+)", color="tab:orange", alpha=0.75)
axis.plot(r_junction[inner_mask], junction_F_eff[inner_mask], linewidth=2.3, color="tab:blue", label="matched F_eff: inner")
axis.plot(r_junction[outer_mask], junction_F_eff[outer_mask], linewidth=2.3, color="tab:orange", label="matched F_eff: outer")

axis.axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
axis.axvline(R_shell, color="black", linestyle="--", linewidth=1.0, alpha=0.8, label="shell")
axis.set_xlabel("r")
axis.set_ylabel("F(r) = r A'(r) - 2 A(r)")
axis.set_title("RN-dS junction photon-sphere diagnostic after shell-time matching")
axis.set_ylim(-4.0, 7.0)
axis.grid(True, alpha=0.25, linewidth=0.6)
mark_photon_spheres(axis, inner_metric, "inner", "tab:blue")
mark_photon_spheres(axis, outer_metric, "outer", "tab:orange")
axis.legend(fontsize=8, loc="best")
plt.show()

## 3. `generate_junction_atlas.py` 的两段式流程

这一节说的“两段式”，本质上是在省计算量：**先用便宜的物理判据扫一大片参数空间，再只挑少数有代表性的点做真正追光成像**。

如果对每个参数点都生成完整图像，成本会很高。因为一个图像不是只算一个公式，而是要对很多屏幕半径 `rho` 追光、找薄盘交点、算红移、采样 profile、再渲染 PNG。所以 atlas 脚本先问一个更便宜的问题：这组参数大概属于哪一类？物理上合不合法？有没有内侧/外侧 photon sphere？

### 第一段：phase map，也就是参数空间地图

`PHASE_MAPS` 定义要扫哪两个参数。RN-dS 这张图叫 `rnds_lambda_shell`，它固定

- `m_minus=0.8`
- `m_plus=1.0`
- `q_minus=q_plus=0.2`
- `lambda_minus=0.01`
- `observer_radius=10.0`

然后扫描二维网格：横轴是 `lambda_plus`，纵轴是 `shell_radius`。对每一个网格点，它只做低成本检查：

1. 用 `AtlasParameters` 组成一组内外 RN-dS 参数。
2. `admit_parameters()` 检查这组参数能不能用于静态 junction，比如壳和观测者是否在静态区、是否有干净的 RN-dS static patch、面压是否有限。
3. `classify_photon_spheres()` 看可用区域里 photon sphere 在壳内、壳外、两边都有，还是都没有。
4. 把结果写成 `phase_maps/rnds_lambda_shell.csv`，再用颜色画成 `phase_maps/rnds_lambda_shell.png`。

所以 phase map 不是图像模拟结果，它更像一张“参数地图”：告诉你哪些参数点可用、哪些不可用、每个可用点属于什么 photon-sphere 结构。

### 第二段：representative cases，也就是挑少数点真正成像

有了 phase map 之后，脚本从 `admitted=True` 的网格点里挑代表 case。挑选依据包括：`inner_only`、`outer_only`、`double`、`large_lambda_jump`、`rnds_observer_near_cosmological_patch` 等标签。挑到的少数 case 才会进入完整流程：

1. 对很多屏幕半径 `rho` 调用 junction ray tracer。
2. 每条 ray 穿过 shell 时用 `match_static_shell()` 重标定 `E` 和 `b`。
3. 找薄盘交点，计算 junction redshift 和观测强度。
4. 得到一维 `I_obs(rho)` profile。
5. 把 profile 旋转成二维图像 PNG。

可以把整个 atlas 理解成：第一段负责“在哪里值得看”，第二段负责“把这些代表点真的看清楚”。

In [ ]:
phase_csv = ROOT / "outputs" / "junction_atlas" / "phase_maps" / "rnds_lambda_shell.csv"

if phase_csv.exists():
    with phase_csv.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    print("phase rows:", len(rows))
    print("admitted:", sum(row["admitted"] == "True" for row in rows))
    counts = {}
    for row in rows:
        key = row["photon_sphere_classification"]
        counts[key] = counts.get(key, 0) + 1
    print("classification counts:", counts)
    print("first row:")
    print(json.dumps(rows[0], indent=2, ensure_ascii=False))
else:
    print("No existing phase CSV found. Generate it with:")
    print("python scripts/generate_junction_atlas.py --families rnds --preset quick --output-dir outputs/junction_atlas")

## 4. 单条 ray 的运行逻辑

对一个屏幕半径 `rho`，完整流程是：

1. `FiniteStaticObserver` 把 `rho` 转成观测角 `alpha`，再转成外侧冲量参数 `b_plus`。
2. `StaticJunctionTransferSolver.trace_screen_radius(rho)` 从外侧观察者开始向内追光。
3. ray 在每个 region 内积分 `dphi/du = 1/sqrt(G(u,b))`。
4. 如果遇到 turning point，径向方向反转。
5. 如果遇到 shell，调用 `match_static_shell()`：

   - `L` 连续。
   - `E_to = E_from * sqrt(A_to(R_shell) / A_from(R_shell))`。
   - 因为 `b=L/E`，所以穿壳后 `b` 会重标定。

6. solver 返回 `JunctionRayResult`，里面有 segments、events、shell_crossings、diagnostics。
7. `compute_intersections()` 在 `phi=pi/2+n*pi` 处找薄盘交点。
8. `annotate_junction_intersections()` 计算 junction redshift `g` 和 `g^4`。
9. `observed_intensity()` 乘以 emissivity，得到该屏幕半径处的总强度。

### 多条光线轨迹：看 shell 处的 refraction

你刚才观察到的现象是对的：前面主线参数 `m_minus=0.8, m_plus=1.0, R_shell=5` 的折射很弱，因为壳面两侧 lapse 很接近。对从外侧进入内侧的光线，

`b_inner / b_outer = sqrt(A_outer(R_shell) / A_inner(R_shell))`。

主线参数里这个比例大约是 `0.93`，也就是局部冲量参数只变了约 `7%`，画在 `x-y` 轨迹图上当然不太显眼。下面我另外取一组仍然是 RN-dS/RN-dS 的静态 junction 参数来专门展示折射：`m_minus=0.2, m_plus=1.0, q_minus=q_plus=0.05, lambda_minus=lambda_plus=0.01, R_shell=3.1`。这组参数没有引入 charge/lambda 跳变，仍然是干净的同族 RN-dS 拼接，但壳面更靠近外侧 photon sphere，且内外质量差更大，所以 `b_inner/b_outer` 约为 `0.62`，折射会明显很多。

图的读法类似论文 Fig. 4 左图：

- 蓝色轨迹：光线穿过 shell，穿壳后 `E` 和 `b=L/E` 按 `A(R_shell)` 比例重标定，然后在另一侧 metric 中继续传播。
- 橙色轨迹：光线没有进入 shell，始终在外侧 RN-dS 区域传播。
- 蓝色虚线圆：薄壳位置 `R_shell`。
- 紫色点线圆：外侧 RN-dS 的宇宙学视界。
- 黑色竖虚线：face-on 薄盘交点方向，对应 `phi=pi/2+n*pi`。

这张图要看的不是某一条 ray 的强度，而是**球壳作为 junction interface 会改变光线后续的局部冲量参数，因此轨迹在穿壳后改由另一套有效势控制**。

补一句容易误解的点：论文里说的 refraction 不是说光线四动量作为几何向量在壳上发生不连续；在壳的局部共动正交标架中，方向可以是连续的。所谓折射主要是相对于两边不同静态 Killing 时间/静态观察者定义的能量、冲量参数和角度发生变化。因此在 `x-y` 轨迹图上不一定表现成特别尖锐的折线，更可靠的诊断是打印 `b_after/b_before` 和穿壳后轨迹由哪一侧 metric 继续控制。


In [ ]:
import warnings
from scipy.integrate import IntegrationWarning

from spherical_raytracing.junction_tracing import StaticJunctionTransferSolver
from spherical_raytracing.policies import SolverOptions

# 主线参数折射弱：这里先把壳面比例直接打印出来。
current_A_inner = junction.inner_metric.A(params.shell_radius)
current_A_outer = junction.outer_metric.A(params.shell_radius)
print("main parameters b_inner/b_outer:", math.sqrt(current_A_outer / current_A_inner))

# 为了看清楚壳上的 refraction，单独构造一组折射更强的 RN-dS/RN-dS junction。
trajectory_params = AtlasParameters(
    family="rnds",
    m_minus=0.2,
    m_plus=1.0,
    q_minus=0.05,
    q_plus=0.05,
    lambda_minus=0.01,
    lambda_plus=0.01,
    shell_radius=3.1,
    observer_radius=10.0,
)
trajectory_admission = admit_parameters(trajectory_params)
trajectory_junction, trajectory_observer = build_junction(trajectory_params)
trajectory_inner = trajectory_junction.inner_metric
trajectory_outer = trajectory_junction.outer_metric
trajectory_R_shell = trajectory_params.shell_radius
trajectory_A_inner = trajectory_inner.A(trajectory_R_shell)
trajectory_A_outer = trajectory_outer.A(trajectory_R_shell)
trajectory_b_factor = math.sqrt(trajectory_A_outer / trajectory_A_inner)
trajectory_E_factor = math.sqrt(trajectory_A_inner / trajectory_A_outer)

print("strong-refraction parameters admitted:", trajectory_admission.admitted)
print("warnings:", trajectory_admission.warnings)
print("A_inner(R_shell):", trajectory_A_inner)
print("A_outer(R_shell):", trajectory_A_outer)
print("outer -> inner E factor sqrt(A_inner/A_outer):", trajectory_E_factor)
print("outer -> inner b factor sqrt(A_outer/A_inner):", trajectory_b_factor)
print("inner photon spheres:", trajectory_inner.photon_spheres())
print("outer photon spheres:", trajectory_outer.photon_spheres())
print("observer radius:", trajectory_observer.r_obs)
print("outer horizons:", trajectory_outer.horizons())
print("outer valid static patch:", trajectory_outer.valid_radial_domain())

# 轨迹图只需要把 ray 追出来并采样 r(phi)。critical_exclusion=0 避免临界附近直接提前停止。
trajectory_solver = StaticJunctionTransferSolver(
    junction=trajectory_junction,
    observer=trajectory_observer,
    options=SolverOptions(max_phi=5.5 * math.pi, critical_exclusion=0.0),
)

def segment_xy(segment, *, samples: int = 90, r_clip: float = math.inf):
    phis = np.linspace(segment.phi_start, segment.phi_end, samples)
    radii = np.array([segment.r_at(float(phi)) for phi in phis], dtype=float)
    keep = np.isfinite(radii) & (radii > 0.0) & (radii <= r_clip)
    return radii[keep] * np.cos(phis[keep]), radii[keep] * np.sin(phis[keep])

trajectory_rhos = [0.8, 1.4, 2.0, 2.6, 3.2, 3.8, 4.4, 5.0, 5.6, 6.2, 6.8, 7.4]
outer_static_patch = trajectory_outer.valid_radial_domain()
outer_cosmological_horizon = outer_static_patch[1] if math.isfinite(outer_static_patch[1]) else None
plot_radius = (outer_cosmological_horizon * 1.04) if outer_cosmological_horizon is not None else trajectory_observer.r_obs * 1.05

figure, axis = plt.subplots(figsize=(6.4, 6.4))
label_used = set()

def once(label: str) -> str | None:
    if label in label_used:
        return None
    label_used.add(label)
    return label

shell_circle = plt.Circle((0.0, 0.0), trajectory_R_shell, fill=False, linestyle="--", linewidth=1.4, color="tab:blue", label="shell")
axis.add_patch(shell_circle)

if outer_cosmological_horizon is not None:
    cosmological_circle = plt.Circle(
        (0.0, 0.0),
        outer_cosmological_horizon,
        fill=False,
        linestyle=":",
        linewidth=1.5,
        color="tab:purple",
        label="outer cosmological horizon",
    )
    axis.add_patch(cosmological_circle)

inner_horizons = [r for r in trajectory_inner.horizons() if 0.0 < r < trajectory_R_shell]
if inner_horizons:
    horizon_circle = plt.Circle((0.0, 0.0), max(inner_horizons), fill=False, linewidth=1.2, color="black", label="inner BH horizon")
    axis.add_patch(horizon_circle)

axis.axvline(0.0, color="black", linestyle="--", linewidth=1.0, alpha=0.55, label="disk direction")
axis.scatter([trajectory_observer.r_obs], [0.0], marker="o", s=36, color="black", zorder=5, label="observer")

trajectory_rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore", IntegrationWarning)
    for rho in trajectory_rhos:
        ray = trajectory_solver.trace_screen_radius(float(rho))
        crosses_shell = len(ray.shell_crossings) > 0
        color = "tab:blue" if crosses_shell else "tab:orange"
        label = "crosses shell" if crosses_shell else "stays outside shell"
        for segment in ray.segments:
            x_values, y_values = segment_xy(segment, r_clip=plot_radius)
            if len(x_values) == 0:
                continue
            axis.plot(x_values, y_values, color=color, linewidth=1.35, alpha=0.82, label=once(label))
        for crossing in ray.shell_crossings:
            axis.scatter(
                [crossing.r * math.cos(crossing.phi)],
                [crossing.r * math.sin(crossing.phi)],
                s=22,
                color=color,
                edgecolor="white",
                linewidth=0.6,
                zorder=6,
                label=once("shell crossing"),
            )
        first_crossing = ray.shell_crossings[0] if ray.shell_crossings else None
        first_b_ratio = None if first_crossing is None else first_crossing.b_after / first_crossing.b_before
        final_event = ray.events[-1] if ray.events else None
        final_radius = None if final_event is None or final_event.u <= 0.0 else 1.0 / final_event.u
        final_label = ray.termination_reason if final_event is None or not final_event.message else final_event.message
        trajectory_rows.append((rho, ray.b_plus, len(ray.shell_crossings), first_b_ratio, final_radius, final_label))

axis.set_aspect("equal", adjustable="box")
axis.set_xlim(-plot_radius, plot_radius)
axis.set_ylim(-plot_radius, plot_radius)
axis.set_xlabel("x = r cos(phi)")
axis.set_ylabel("y = r sin(phi)")
axis.set_title("RN-dS static-shell junction ray trajectories: stronger refraction example")
axis.grid(True, alpha=0.22, linewidth=0.6)
axis.legend(fontsize=8, loc="upper left")
plt.show()

for rho, b_plus, crossing_count, first_b_ratio, final_radius, termination in trajectory_rows:
    ratio_text = "--" if first_b_ratio is None else f"{first_b_ratio:.3f}"
    radius_text = "--" if final_radius is None else f"{final_radius:.3f}"
    print(
        f"rho={rho:4.1f}  b_plus={b_plus:7.3f}  shell_crossings={crossing_count}  "
        f"first b_after/b_before={ratio_text}  r_final={radius_text}  termination={termination}"
    )


In [ ]:
from spherical_raytracing.junction_tracing import StaticJunctionTransferSolver
from spherical_raytracing.junctions import annotate_junction_intersections
from spherical_raytracing.transfer import DiskWindow, compute_intersections
from spherical_raytracing.sources import ThinDiskSource, observed_intensity

# transfer solver 是图像生产的主后端；Hamiltonian 后端主要用于对照诊断。
solver = StaticJunctionTransferSolver(junction=junction, observer=observer)
# 薄盘允许出现在壳内外两侧，所以 enabled_regions 同时包含 inner 和 outer。
disk = DiskWindow(r_min=0.0, r_max=max(params.observer_radius * 2.0, 100.0), enabled_regions=frozenset({"inner", "outer"}))
# 这里用简单的 1/r^2 发射率；atlas 脚本还支持 paper-style emissivity。
source = ThinDiskSource(lambda r, region=None: 1.0 / max(r**2, 1e-300))

def trace_one_rho(rho: float, max_order: int = 4):
    # rho 是屏幕半径。solver 内部会把 rho -> alpha -> b_plus，然后从外侧观测者向内追光。
    ray = solver.trace_screen_radius(rho)
    # ray.segments 记录了每一段在哪个 region、从哪个 phi 到哪个 phi，因此可以反查薄盘交点。
    intersections = compute_intersections(ray, disk, max_order=max_order)
    # junction 红移需要考虑交点所在 region 以及此前经历过的壳穿越能量重标定。
    annotated = annotate_junction_intersections(
        intersections,
        ray,
        observer,
        {"inner": junction.inner_metric, "outer": junction.outer_metric},
    )
    # observed.total 就是这个屏幕半径 rho 上的一维图像强度 I_obs(rho)。
    observed = observed_intensity(
        annotated,
        source,
        junction.outer_metric,
        observer,
        region_metrics={"inner": junction.inner_metric, "outer": junction.outer_metric},
    )
    return ray, annotated, observed

# 试几个屏幕半径，挑第一个真的打到薄盘的样本，方便观察完整数据结构。
candidate_rhos = [0.5, 1.0, 2.0, 4.0, 6.0]
chosen = None
for rho in candidate_rhos:
    ray, annotated, observed = trace_one_rho(rho)
    if annotated:
        chosen = rho
        break

if chosen is None:
    chosen = candidate_rhos[0]
    ray, annotated, observed = trace_one_rho(chosen)

print("chosen rho:", chosen)
print("b_plus:", ray.b_plus)
print("termination:", ray.termination_reason)
print("segments:", [(s.region, s.radial_direction, s.endpoint_event.value, s.phi_start, s.phi_end) for s in ray.segments])
print("shell crossings:", len(ray.shell_crossings))
for crossing in ray.shell_crossings:
    print("  crossing:", crossing)
print("intersections:")
for item in annotated:
    print({
        "m": item.m,
        "r": item.r,
        "phi": item.phi,
        "region": item.region,
        "path_class": item.path_class,
        "g": item.diagnostics.get("g"),
        "redshift_weight": item.diagnostics.get("redshift_weight"),
    })
print("observed total intensity:", observed.total)

## 5. 从很多条 ray 到一维 profile

atlas 不会只追一条 ray，而是沿屏幕半径 `rho` 追很多条 ray。核心函数是 `sample_radial_profile()`。

你的理解基本是对的：第一步确实是在 `rho_min` 到 `rho_max` 上均匀采样；第二步看相邻两个样本之间有没有剧烈变化，如果有，就在这个小区间中点再补一条 ray。这个过程重复 `max_refine` 轮。

加密采样主要看这些信号：

- 强度梯度太大。
- 相邻样本强度相对跳变太大。
- termination reason 改变。
- path class 改变。
- shell crossing 数改变。
- backend comparison 出现 disagreement。

atlas 版 `_trace_profile()` 还会把 photon sphere 的临界屏幕半径、内侧 photon sphere 穿壳映射后的屏幕半径、以及一些 focused sampling 点预先塞进采样列表。

注意：采样坐标是屏幕上的 `rho`，但物理上更常用的横坐标是外侧守恒冲量参数 `b_plus`。有限半径观测者处二者单调相关，但不是同一个量；代码里每条 ray 都会先把 `rho -> alpha -> b_plus`。

为了让下面这张一维曲线更光滑，我这里不用 atlas 的 `rho` 自适应样本直接画图，而是**直接在 `b_plus` 上等距高分辨率采样**。每个 `b_plus` 调用 `solver.trace_b(b_plus)` 追一条 ray；同时把对应的 `rho(b_plus)` 存下来，这样下一节二维图像仍然可以按屏幕半径 `rho` 插值。

这一步解决的是“画图采样不均匀”造成的锯齿；如果曲线在某些位置仍有尖峰或折点，那通常对应 photon-ring 附近的高阶交点、termination 改变、或者 shell crossing 数改变，不应该简单用平滑滤波抹掉。
为了做基线比较，本节还会画单一时空的 Schwarzschild、RN、RN-dS profile。参数尽量贴近拼接时空的外侧，因为观测者和 `b_plus` 本来就是外侧定义：Schwarzschild 取 `M=m_plus`，RN 取 `(M,Q)=(m_plus,q_plus)`，纯 RN-dS 取 `(M,Q,Lambda)=(m_plus,q_plus,lambda_plus)`。所有单一时空都用同一个有限半径观测者 `r_obs=observer_radius`、同一个屏幕窗口 `rho_max=8`，薄盘窗口和发射率也保持一致：`DiskWindow(r_min=0, r_max=100)` 与 `I_em(r)=1/r^2`。单一时空没有 shell，所以横坐标直接记作 `b`。

In [ ]:
import math
import warnings
import numpy as np
from scipy.integrate import IntegrationWarning
from spherical_raytracing.imaging import RadialProfile
from spherical_raytracing.metrics import ReissnerNordstromDeSitterMetric, ReissnerNordstromMetric, SchwarzschildMetric
from spherical_raytracing.observers import FiniteStaticObserver
from spherical_raytracing.policies import SolverOptions
from spherical_raytracing.solvers import QuadTransferSolver

# 这里专门按 b_plus 等距采样；rho 只作为下一节渲染屏幕图像时的插值坐标保存。
rho_max_for_plot = 8.0
b_max_for_plot = observer.impact_parameter(math.atan(rho_max_for_plot / observer.r_obs))
b_sample_count = 1000
max_order_for_profile = 4
b_grid = np.linspace(0.0, b_max_for_plot, b_sample_count)
rows_by_bplus = {}
profile_solver = StaticJunctionTransferSolver(
    junction=junction,
    observer=observer,
    options=SolverOptions(critical_exclusion=0.0),
)

def rho_from_bplus(b_plus: float) -> float:
    if b_plus <= 0.0:
        return 0.0
    sine = b_plus * math.sqrt(observer.metric.A(observer.r_obs)) / observer.r_obs
    alpha = math.asin(min(max(sine, 0.0), 0.999999999))
    return observer.screen_radius(alpha)

def trace_one_bplus(b_plus: float, max_order: int = 3):
    b_plus = float(b_plus)
    if b_plus <= 0.0:
        row = {"rho": 0.0, "b_plus": 0.0, "intensity": 0.0, "termination_reason": "axis", "shell_crossing_count": 0, "intersection_count": 0, "order_intensities": {order: 0.0 for order in range(1, max_order + 1)}}
        rows_by_bplus[b_plus] = row
        return row

    ray = profile_solver.trace_b(b_plus)
    intersections = compute_intersections(ray, disk, max_order=max_order)
    annotated = annotate_junction_intersections(
        intersections,
        ray,
        observer,
        {"inner": junction.inner_metric, "outer": junction.outer_metric},
    )
    observed = observed_intensity(
        annotated,
        source,
        junction.outer_metric,
        observer,
        region_metrics={"inner": junction.inner_metric, "outer": junction.outer_metric},
    )
    order_intensities = {order: 0.0 for order in range(1, max_order + 1)}
    for item, contribution in zip(annotated, observed.contributions):
        order_intensities[item.m] += float(contribution)
    row = {
        "rho": rho_from_bplus(b_plus),
        "b_plus": b_plus,
        "intensity": observed.total,
        "termination_reason": ray.termination_reason,
        "shell_crossing_count": len(ray.shell_crossings),
        "intersection_count": len(annotated),
        "order_intensities": order_intensities,
    }
    rows_by_bplus[b_plus] = row
    return row

with warnings.catch_warnings():
    # 个别接近临界/转折的 ray 会让 quad 报 roundoff warning；这里保留结果但不让输出淹没图像。
    warnings.simplefilter("ignore", IntegrationWarning)
    profile_rows = [trace_one_bplus(b_plus, max_order=max_order_for_profile) for b_plus in b_grid]
b_values = np.array([row["b_plus"] for row in profile_rows], dtype=float)
rho_values = np.array([row["rho"] for row in profile_rows], dtype=float)
intensity_values = np.array([row["intensity"] for row in profile_rows], dtype=float)

# bplus_profile 用来画这一节的物理曲线；profile 保持 rho 坐标，供下一节 render_axisymmetric_image 使用。
bplus_profile = RadialProfile(
    coordinates=b_values,
    intensities=intensity_values,
    diagnostics={"sample_count": len(b_values), "coordinate": "b_plus"},
)
rho_order = np.argsort(rho_values)
profile = RadialProfile(
    coordinates=rho_values[rho_order],
    intensities=intensity_values[rho_order],
    diagnostics={"sample_count": len(rho_values), "coordinate": "rho", "source_sampling": "uniform_b_plus"},
)

termination_counts = {}
for row in profile_rows:
    key = row["termination_reason"]
    termination_counts[key] = termination_counts.get(key, 0) + 1

print("b_plus sample count:", len(profile_rows))
print("b_plus range:", (float(b_values[0]), float(b_values[-1])))
print("corresponding rho range:", (float(rho_values[0]), float(rho_values[-1])))
print("termination counts:", termination_counts)
print("first rows:")
for row in profile_rows[:5]:
    print(row)

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(b_values, intensity_values, color="black", linewidth=1.5, alpha=0.9)
    ax.set_xlabel(r"$b_+$")
    ax.set_ylabel(r"total observed intensity $I_{obs}$")
    ax.set_title(r"Uniform-$b_+$ profile $I_{obs}(b_+)$")
    ax.grid(True, alpha=0.25)
    plt.show()
except Exception as exc:
    print("Matplotlib display skipped:", exc)

def sample_single_metric_profile(metric, *, label: str):
    # 单一时空共用这条路径：metric -> finite observer -> QuadTransferSolver -> disk intersections -> intensity。
    single_observer = FiniteStaticObserver(r_obs=params.observer_radius, metric=metric)
    single_solver = QuadTransferSolver(
        metric=metric,
        observer=single_observer,
        options=SolverOptions(critical_exclusion=0.0),
    )
    single_disk = DiskWindow(r_min=disk.r_min, r_max=disk.r_max, enabled_regions=frozenset({metric.region}))
    b_max = single_observer.impact_parameter(math.atan(rho_max_for_plot / single_observer.r_obs))
    b_grid_single = np.linspace(0.0, b_max, b_sample_count)

    def trace_single_b(b: float):
        b = float(b)
        if b <= 0.0:
            return {"b": 0.0, "intensity": 0.0, "termination_reason": "axis", "intersection_count": 0, "order_intensities": {order: 0.0 for order in range(1, max_order_for_profile + 1)}}
        ray = single_solver.trace_b(b)
        intersections = compute_intersections(ray, single_disk, max_order=max_order_for_profile)
        observed = observed_intensity(intersections, source, metric, single_observer)
        order_intensities = {order: 0.0 for order in range(1, max_order_for_profile + 1)}
        for item, contribution in zip(intersections, observed.contributions):
            order_intensities[item.m] += float(contribution)
        return {
            "b": b,
            "intensity": observed.total,
            "termination_reason": ray.diagnostics.termination_reason,
            "intersection_count": len(intersections),
            "order_intensities": order_intensities,
        }

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", IntegrationWarning)
        rows = [trace_single_b(b) for b in b_grid_single]

    b_values_single = np.array([row["b"] for row in rows], dtype=float)
    intensity_values_single = np.array([row["intensity"] for row in rows], dtype=float)
    profile_single = RadialProfile(
        coordinates=b_values_single,
        intensities=intensity_values_single,
        diagnostics={"sample_count": len(b_values_single), "coordinate": "b", "label": label},
    )
    termination_counts_single = {}
    for row in rows:
        key = row["termination_reason"]
        termination_counts_single[key] = termination_counts_single.get(key, 0) + 1
    print(f"{label} b sample count:", len(rows))
    print(f"{label} b range:", (float(b_values_single[0]), float(b_values_single[-1])))
    print(f"{label} termination counts:", termination_counts_single)
    return rows, profile_single, b_values_single, intensity_values_single, termination_counts_single

def plot_single_metric_profile(b_values_single, intensity_values_single, *, title: str):
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(b_values_single, intensity_values_single, color="black", linewidth=1.5, alpha=0.9)
        ax.set_xlabel(r"$b$")
        ax.set_ylabel(r"total observed intensity $I_{obs}$")
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        plt.show()
    except Exception as exc:
        print("Matplotlib display skipped:", exc)

def profile_order_components(rows):
    direct = np.array([row["order_intensities"].get(1, 0.0) for row in rows], dtype=float)
    lensing = np.array([row["order_intensities"].get(2, 0.0) for row in rows], dtype=float)
    photon = np.array([
        sum(value for order, value in row["order_intensities"].items() if order >= 3)
        for row in rows
    ], dtype=float)
    total = direct + lensing + photon
    return direct, lensing, photon, total

def plot_order_decomposition(b_axis, rows, *, title: str, xlabel: str = r"$b$"):
    direct, lensing, photon, total_from_orders = profile_order_components(rows)
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(7.2, 4.2))
        ax.plot(b_axis, total_from_orders, color="black", linewidth=1.7, label="total")
        ax.plot(b_axis, direct, linewidth=1.2, label="m=1 direct")
        ax.plot(b_axis, lensing, linewidth=1.2, label="m=2 lensing ring")
        ax.plot(b_axis, photon, linewidth=1.2, label="m>=3 photon ring")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(r"observed intensity contribution")
        ax.set_title(title)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=8)
        plt.show()
    except Exception as exc:
        print("Order decomposition plot skipped:", exc)

# Schwarzschild、RN、纯 RN-dS 对照：都使用外侧参数，方便和 junction 的 b_plus 直接比较。
schwarzschild_metric = SchwarzschildMetric(mass=params.m_plus, region="external")
rn_metric = ReissnerNordstromMetric(mass=params.m_plus, charge=params.q_plus, region="external")
rnds_metric = ReissnerNordstromDeSitterMetric(
    mass=params.m_plus,
    charge=params.q_plus,
    cosmological_constant=params.lambda_plus,
    region="external",
)

schwarzschild_rows, schwarzschild_profile, schwarzschild_b_values, schwarzschild_intensity_values, schwarzschild_termination_counts = sample_single_metric_profile(schwarzschild_metric, label="Schwarzschild")
plot_single_metric_profile(schwarzschild_b_values, schwarzschild_intensity_values, title=r"Schwarzschild profile $I_{obs}(b)$ with the same disk emissivity")

rn_rows, rn_profile, rn_b_values, rn_intensity_values, rn_termination_counts = sample_single_metric_profile(rn_metric, label="RN")
plot_single_metric_profile(rn_b_values, rn_intensity_values, title=r"RN profile $I_{obs}(b)$ with outer parameters $(M,Q)=(m_+,q_+)$")
plot_order_decomposition(rn_b_values, rn_rows, title=r"RN profile decomposed by disk-intersection order $m$")
# 同样的分阶思想也可以用在 junction profile 上，检查壳是否主要改变 direct、lensing 还是 photon-ring 分量。
plot_order_decomposition(b_values, profile_rows, title=r"RN-dS junction profile decomposed by disk-intersection order $m$", xlabel=r"$b_+$")

rnds_rows, rnds_profile, rnds_b_values, rnds_intensity_values, rnds_termination_counts = sample_single_metric_profile(rnds_metric, label="RN-dS")
plot_single_metric_profile(rnds_b_values, rnds_intensity_values, title=r"Pure RN-dS profile $I_{obs}(b)$ with outer parameters $(M,Q,\Lambda)=(m_+,q_+,\Lambda_+)$")

# 更直接的诊断图：四条曲线叠加，以及 junction 相对纯 RN-dS 的差值。
try:
    import matplotlib.pyplot as plt

    common_b_max = min(b_values[-1], schwarzschild_b_values[-1], rn_b_values[-1], rnds_b_values[-1])
    common_b = np.linspace(0.0, common_b_max, b_sample_count)
    junction_common = np.interp(common_b, b_values, intensity_values)
    schwarzschild_common = np.interp(common_b, schwarzschild_b_values, schwarzschild_intensity_values)
    rn_common = np.interp(common_b, rn_b_values, rn_intensity_values)
    rnds_common = np.interp(common_b, rnds_b_values, rnds_intensity_values)

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(common_b, schwarzschild_common, label="Schwarzschild", linewidth=1.2)
    ax.plot(common_b, rn_common, label="RN", linewidth=1.2)
    ax.plot(common_b, rnds_common, label="pure RN-dS", linewidth=1.2)
    ax.plot(common_b, junction_common, label="RN-dS junction", linewidth=1.8, color="black")
    ax.set_xlabel(r"$b$ or $b_+$")
    ax.set_ylabel(r"total observed intensity $I_{obs}$")
    ax.set_title("Profile comparison under the same observer and disk model")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    plt.show()

    fig, ax = plt.subplots(figsize=(7.2, 3.6))
    ax.plot(common_b, junction_common - rnds_common, color="black", linewidth=1.4)
    ax.axhline(0.0, color="gray", linewidth=0.8)
    ax.set_xlabel(r"$b$ or $b_+$")
    ax.set_ylabel(r"$I_{junction}-I_{pure\ RN-dS}$")
    ax.set_title("Shell-induced profile difference relative to pure RN-dS")
    ax.grid(True, alpha=0.25)
    plt.show()
except Exception as exc:
    print("Comparison plots skipped:", exc)

### junction 中哪个 photon sphere 在控制 profile 峰？

这组参数下，RN-dS junction **有效地只有一个 photon sphere** 会控制 profile 中的尖峰：它是**内侧 RN-dS 时空的 photon sphere**。

原因是 junction 把流形切成两块：内侧区域只保留 `r <= R_shell`，外侧区域只保留 `r >= R_shell`。当前 `R_shell=5`，外侧 RN-dS 自己的 photon sphere 在 `r≈2.97`，落在壳里面，所以它在拼接后的外侧区域中已经被切掉了；内侧 RN-dS 的 photon sphere 在 `r≈2.37`，落在壳内，所以它是真正存在于 junction 几何中的 photon sphere。

但是 profile 横坐标是外侧观测者定义的 `b_+`，不是内侧局部的 `b_-`。光从外侧穿壳进入内侧时

`b_- = b_+ sqrt(A_+(R_shell) / A_-(R_shell))`。

所以内侧 photon sphere 的局部临界值 `b_-^crit` 映射到外侧图像横坐标时变成

`b_+^mapped = b_-^crit sqrt(A_-(R_shell) / A_+(R_shell))`。

这就是为什么 junction profile 的峰对应内侧 photon sphere，但出现在一个和纯 RN-dS 的 `b_crit` 不同的 `b_+` 位置。

In [ ]:
def critical_b(metric, r_ph: float) -> float:
    return r_ph / math.sqrt(metric.A(r_ph))

R_shell = params.shell_radius
A_inner_shell = junction.inner_metric.A(R_shell)
A_outer_shell = junction.outer_metric.A(R_shell)
inner_to_outer_b_factor = math.sqrt(A_inner_shell / A_outer_shell)
outer_to_inner_b_factor = math.sqrt(A_outer_shell / A_inner_shell)

inner_effective = []
for r_ph in junction.inner_metric.photon_spheres():
    b_local = critical_b(junction.inner_metric, r_ph)
    b_plus_mapped = b_local * inner_to_outer_b_factor
    inner_effective.append((r_ph, b_local, b_plus_mapped, r_ph <= R_shell))

outer_effective = []
for r_ph in junction.outer_metric.photon_spheres():
    b_local = critical_b(junction.outer_metric, r_ph)
    outer_effective.append((r_ph, b_local, r_ph >= R_shell))

print("shell radius R:", R_shell)
print("A_inner(R):", A_inner_shell)
print("A_outer(R):", A_outer_shell)
print("outer -> inner b factor sqrt(A_outer/A_inner):", outer_to_inner_b_factor)
print("inner critical b -> observed b_plus factor sqrt(A_inner/A_outer):", inner_to_outer_b_factor)
print()
print("inner RN-dS photon spheres: r_ph, local b_minus_crit, mapped b_plus, survives r<=R")
for row in inner_effective:
    print(row)
print("outer RN-dS photon spheres: r_ph, local b_plus_crit, survives r>=R")
for row in outer_effective:
    print(row)

effective_count = sum(1 for _, _, _, survives in inner_effective if survives) + sum(1 for _, _, survives in outer_effective if survives)
print("effective photon-sphere count in this junction:", effective_count)

inner_mapped_b_values = [b_plus for _, _, b_plus, survives in inner_effective if survives]
outer_surviving_b_values = [b_plus for _, b_plus, survives in outer_effective if survives]
outer_cut_b_values = [b_plus for _, b_plus, survives in outer_effective if not survives]

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.plot(b_values, intensity_values, color="black", linewidth=1.6, label="RN-dS junction")
    ax.plot(rnds_b_values, rnds_intensity_values, color="tab:blue", linewidth=1.2, alpha=0.85, label="pure RN-dS outer parameters")
    for value in inner_mapped_b_values:
        ax.axvline(value, color="tab:red", linestyle="--", linewidth=1.2, label="inner photon sphere mapped to b_plus")
    for value in outer_cut_b_values:
        ax.axvline(value, color="tab:blue", linestyle=":", linewidth=1.2, label="outer photon sphere of pure RN-dS, cut out by shell")
    for value in outer_surviving_b_values:
        ax.axvline(value, color="tab:green", linestyle="-.", linewidth=1.2, label="surviving outer photon sphere")
    ax.set_xlabel(r"$b$ or $b_+$")
    ax.set_ylabel(r"total observed intensity $I_{obs}$")
    ax.set_title("Which photon sphere sets the junction profile peak?")
    ax.grid(True, alpha=0.25)
    handles, labels = ax.get_legend_handles_labels()
    unique = dict(zip(labels, handles))
    ax.legend(unique.values(), unique.keys(), fontsize=8)
    plt.show()
except Exception as exc:
    print("Photon-sphere diagnostic plot skipped:", exc)

## 6. 从一维 profile 到二维图像

图像生成本身非常直接：因为这里假设 face-on、轴对称，屏幕上每个像素只需要计算径向坐标，再从一维 `RadialProfile` 插值得到强度。

也就是说，二维图像这一步**不会重新对每个像素追光**。真正昂贵的追光已经在第 5 节完成了：很多个 `rho` 或 `b_plus` 样本已经被转换成一维强度 profile。第 6 节只是把这条径向曲线绕屏幕中心旋转成二维轴对称图像。

这里有两种常用画法：

- **屏幕 `rho` 图像**：像素半径是 `rho_pixel=sqrt(x^2+y^2)`，因此要用 `profile`，也就是 `coordinates=rho_values` 的 profile。
- **冲量参数 `b`/`b_+` 图像**：像素半径直接解释成 `b_pixel=sqrt(b_x^2+b_y^2)` 或 `b_{+,pixel}`，因此可以直接用上一节的 `bplus_profile`、`schwarzschild_profile`、`rn_profile`、`rnds_profile`。

你现在要看的四种情况对比，我下面采用第二种：**直接用 `b`/`b_+` profile 旋转成二维图像**。这样四张图的坐标都是冲量参数平面，更方便比较 photon-ring 位置。

`render_axisymmetric_image(profile, ImageGrid(width, height, radius))` 的内部逻辑是：

1. 在 `[-radius, radius]` 上生成 `x`、`y` 网格。
2. 用 `meshgrid` 得到每个像素的 `(x,y)`。
3. 计算 `rr=sqrt(x^2+y^2)`。
4. 调用 `profile.evaluate(rr)`，用一维 profile 的 PCHIP 插值给每个像素赋强度。
5. 返回 `RenderedImage(x, y, pixels)`；后面 `imshow()` 只是把 `pixels` 显示成图。

因此二维图像的环、暗区、亮峰都不是额外物理计算出来的新对象，而是一维 intensity profile 的二维旋转表示。

In [ ]:
from spherical_raytracing.imaging import ImageGrid, render_axisymmetric_image

# 这里直接使用 b/b_plus profile，而不是 rho profile。
# 得到的是冲量参数平面上的轴对称图：像素半径代表 b 或 b_plus。
image_profiles = [
    ("Schwarzschild", schwarzschild_profile, r"$b$"),
    ("RN", rn_profile, r"$b$"),
    ("pure RN-dS", rnds_profile, r"$b$"),
    ("RN-dS junction", bplus_profile, r"$b_+$"),
]

rendered_images = {}
for name, radial_profile, coordinate_label in image_profiles:
    radius = float(radial_profile.coordinates[-1])
    image_grid = ImageGrid(width=180, height=180, radius=radius)
    rendered = render_axisymmetric_image(radial_profile, image_grid)
    rendered_images[name] = rendered
    print(f"{name}: coordinate={radial_profile.diagnostics.get('coordinate')}, radius={radius:.4f}, pixel min/max=({float(np.min(rendered.pixels)):.4g}, {float(np.max(rendered.pixels)):.4g})")

try:
    import matplotlib.pyplot as plt

    global_vmax = max(float(np.max(image.pixels)) for image in rendered_images.values())
    fig, axes = plt.subplots(2, 2, figsize=(8, 8), constrained_layout=True)
    for axis, (name, radial_profile, coordinate_label) in zip(axes.flat, image_profiles):
        image = rendered_images[name]
        radius = float(radial_profile.coordinates[-1])
        extent = [-radius, radius, -radius, radius]
        axis.imshow(
            image.pixels,
            origin="lower",
            cmap="afmhot",
            vmin=0.0,
            vmax=max(global_vmax, 1e-300),
            extent=extent,
        )
        axis.set_title(name)
        axis.set_xlabel(coordinate_label)
        axis.set_ylabel(coordinate_label)
        axis.set_aspect("equal")
    fig.suptitle("Images rendered directly from b / b_plus profiles")
    plt.show()
except Exception as exc:
    print("Matplotlib display skipped:", exc)

## 7. atlas 输出目录怎么看

如果已经运行过 atlas，RN-dS 相关输出主要在：

- `outputs/junction_atlas/phase_maps/rnds_lambda_shell.csv`
- `outputs/junction_atlas/phase_maps/rnds_lambda_shell.png`
- `outputs/junction_atlas/cases/rnds_*/profile_paper.json`
- `outputs/junction_atlas/cases/rnds_*/profile_paper.csv`
- `outputs/junction_atlas/cases/rnds_*/profile_paper.png`
- `outputs/junction_atlas/cases/rnds_*/image_paper.png`
- `outputs/junction_atlas/cases/rnds_*/transfer_redshift.png`
- `outputs/junction_atlas/figures/fig_case_rnds_*.png`

`manifest.json` 是总索引，报告脚本 `write_junction_atlas_report.py` 就是读它和各 case 文件来写中文说明。

In [ ]:
# manifest 是 atlas 的总索引：phase map 摘要、被选中的代表 case、每张图的位置都在这里。
manifest_path = ROOT / "outputs" / "junction_atlas" / "manifest.json"

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("preset:", manifest.get("preset"))
    print("families:", manifest.get("families"))
    print("selected cases:", len(manifest.get("selected_case_summaries", [])))
    # selected_case_summaries 只列代表性案例；完整网格信息在 phase_maps/*.csv。
    rnds_cases = [case for case in manifest.get("selected_case_summaries", []) if case.get("family") == "rnds"]
    print("selected RN-dS cases:", len(rnds_cases))
    for case in rnds_cases[:5]:
        primary = case.get("primary_emissivity", "paper")
        image_path = ROOT / "outputs" / "junction_atlas" / case["case_dir"] / f"image_{primary}.png"
        profile_path = ROOT / "outputs" / "junction_atlas" / case["case_dir"] / f"profile_{primary}.json"
        print("-", case["case_id"], case.get("category_tags"), image_path.relative_to(ROOT), "exists=", image_path.exists())
        if profile_path.exists():
            profile_payload = json.loads(profile_path.read_text())
            print("  sample_count:", len(profile_payload.get("samples", [])))
            print("  sampling diagnostics:", profile_payload.get("sampling_diagnostics", {}))
else:
    print("No manifest found. Generate one with:")
    print("python scripts/generate_junction_atlas.py --preset quick --families rn rnds --output-dir outputs/junction_atlas")

In [ ]:
# 如果当前环境支持 IPython display，就直接把已有 RN-dS 图和 transfer/redshift 图显示出来。
try:
    from IPython.display import Image, display
    if manifest_path.exists():
        for case in rnds_cases[:3]:
            primary = case.get("primary_emissivity", "paper")
            image_path = ROOT / "outputs" / "junction_atlas" / case["case_dir"] / f"image_{primary}.png"
            redshift_path = ROOT / "outputs" / "junction_atlas" / case["case_dir"] / "transfer_redshift.png"
            print(case["case_id"], case.get("category_tags"))
            if image_path.exists():
                display(Image(filename=str(image_path)))
            if redshift_path.exists():
                display(Image(filename=str(redshift_path)))
except Exception as exc:
    print("Image display skipped:", exc)

## 8. 常用生成命令

最小 smoke test：

```bash
python scripts/generate_junction_atlas.py \
  --preset quick \
  --families rn rnds \
  --output-dir outputs/junction_atlas
```

更接近报告尺度，并比较 transfer/Hamiltonian 后端：

```bash
python scripts/generate_junction_atlas.py \
  --preset paper \
  --families rn rnds \
  --output-dir outputs/junction_atlas \
  --compare-backends \
  --include-schwarzschild-reference \
  --emissivity both
```

写中文 atlas 报告：

```bash
python scripts/write_junction_atlas_report.py \
  --manifest outputs/junction_atlas/manifest.json \
  --output docs/junction-atlas/rn-rnds-static-junction-atlas.md
```

## 9. 关键概念回收

- **RN-dS 的边界不是无穷远**：正 `Lambda` 下有宇宙学视界，观测者必须是有限半径静态观测者。
- **clean static patch 很重要**：RN-dS junction 要求壳和外侧观测者在黑洞外视界与宇宙学视界之间。
- **壳不是简单透明界面**：穿壳时 `L` 连续，但 `E` 和 `b=L/E` 会按 `A(R_shell)` 比例重标定。
- **图像来自一维 profile**：每个 `rho` 一条 ray，得到 `I_obs(rho)`；二维图像只是轴对称旋转。
- **phase map 和 image 是两种成本层级**：phase map 只做几何和物理筛选；代表 case 才做完整追光、profile、PNG。
- **Hamiltonian backend 是诊断工具**：默认图像生产主路径是 transfer solver，Hamiltonian 用于交叉比较。